# COMP8410 A1 - Week 5 (TRELLIS.2 vs Hunyuan3D 2.1, single image to 3D) - Colab
**Two sessions.** Part A on a cheap **T4** runtime builds both environments and packs them to Drive. Part B on an **A100** unpacks them and does all generation and scoring. If Part A is skipped, Cell B1 builds the environments on the A100 instead (works, costs more).
Before running: Hugging Face account with an access token (read), and accepted terms for the gated models the TRELLIS.2 pipeline loads (Cell B3 prints which ones). Photos: Drive `comp8410_a1/w5_data/setC/`.
Before saving: tick *Omit code cell output when saving*. **Disconnect the runtime when finished** (last cell).

## Part A (T4 runtime): shared setup, then build and pack the environments

In [ ]:
%cd /content
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs', 'w5_data', 'w5_envs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!pip -q install -r requirements.txt
!pip -q install trimesh scipy tabulate rembg onnxruntime transformers lpips
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# A1. micromamba + TRELLIS.2 environment (long: compiles CUDA extensions). Read the last lines of the output.
!bash pairs/w5_image_to_3d/install_envs.sh micromamba
!bash pairs/w5_image_to_3d/install_envs.sh trellis2 2>&1 | tee {BASE}/logs/w5_install_trellis2.log | tail -40

In [ ]:
# A2. Hunyuan3D 2.1 environment. The last lines print the licence text: read it (territory and use limits) for the report.
!bash pairs/w5_image_to_3d/install_envs.sh hunyuan 2>&1 | tee {BASE}/logs/w5_install_hunyuan.log | tail -60

In [ ]:
# A3. pack both environments to Drive (needs roughly 10-20 GB free) so the A100 session can skip A1-A2
!bash pairs/w5_image_to_3d/install_envs.sh pack {BASE}/w5_envs

**Now disconnect the T4 runtime and connect an A100 runtime.** Then continue at Part B.

## Part B (A100 runtime)
### Cell B1: Drive, code, packages, environments

In [ ]:
%cd /content
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs', 'w5_data']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!pip -q install -r requirements.txt
!pip -q install trimesh scipy tabulate rembg onnxruntime transformers lpips
!nvidia-smi --query-gpu=name,memory.total --format=csv
if os.path.exists(f'{BASE}/w5_envs/w5_envs.tar.zst'):
    !bash pairs/w5_image_to_3d/install_envs.sh unpack {BASE}/w5_envs
else:
    print('no packed environments: building them here (slow, billed at the A100 rate)')
    !bash pairs/w5_image_to_3d/install_envs.sh micromamba
    !bash pairs/w5_image_to_3d/install_envs.sh trellis2 2>&1 | tail -30
    !bash pairs/w5_image_to_3d/install_envs.sh hunyuan 2>&1 | tail -30

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!rm -rf results outputs
!ln -s {BASE}/results results
!ln -s {BASE}/outputs outputs
!mkdir -p {BASE}/outputs/w5_figures
!python -m pytest -q tests/test_w5.py

### Cell B2: Hugging Face login
Paste the token when asked (it is not saved in the notebook).

In [ ]:
import getpass, os
from huggingface_hub import login
tok = getpass.getpass('HF token: ')
login(token=tok)
os.environ['HF_TOKEN'] = tok
os.environ['HF_HOME'] = '/content/hf'

### Cell B3: which gated models does TRELLIS.2 need, and do you have access?

In [ ]:
import re, json
from huggingface_hub import hf_hub_download, HfApi
cfg = hf_hub_download('microsoft/TRELLIS.2-4B', 'pipeline.json')
txt = open(cfg).read()
repos = sorted(set(re.findall(r'"([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)"', txt)))
print('repos named in pipeline.json:', repos)
api = HfApi()
for r in repos:
    try:
        info = api.model_info(r)
        print('OK   ', r, '| gated:', info.gated)
        api.auth_check(r)
    except Exception as e:
        print('FAIL ', r, '->', str(e)[:120], ' (open the model page and accept the terms)')

### Cell B4: inputs. Clean copies of the Set C photos, background removal, contact sheet. LOOK at the sheet before going on.

In [ ]:
!python pairs/w5_image_to_3d/prepare_inputs.py --raw {BASE}/w5_data/setC
!python pairs/w5_image_to_3d/run_w5.py prep 2>&1 | tail -20
from PIL import Image
display(Image.open('outputs/w5_figures/inputs_sheet.jpg'))

## Cell C: smoke tests (1 item, 1 seed per model). Results go to separate smoke files.
Each model has its own environment, so each runs through `micromamba run`. After both, the last cell scores them and shows the view sheet: check the object stands upright and the textures look right. If a model lies on its side, change `UP_AXIS` in `w5_common.py`.

In [ ]:
!cd /content/TRELLIS.2 && /content/bin/micromamba run -n trellis2 env MAMBA_ROOT_PREFIX=/content/mamba python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_trellis2.py" --items c01_topiary --seeds 0 --out-root /content/smoke_out --results "/content/a1/48576158 Evaluation/results/w5_smoke.jsonl" 2>&1 | tee {BASE}/logs/w5_smoke_trellis2.log | grep -v Warning | tail -30

In [ ]:
!cd /content/Hunyuan3D-2.1 && /content/bin/micromamba run -n hunyuan env MAMBA_ROOT_PREFIX=/content/mamba python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_hunyuan3d.py" --items c01_topiary --seeds 0 --out-root /content/smoke_out --results "/content/a1/48576158 Evaluation/results/w5_smoke.jsonl" 2>&1 | tee {BASE}/logs/w5_smoke_hunyuan.log | grep -v Warning | tail -30

In [ ]:
!python pairs/w5_image_to_3d/score_w5.py --items c01_topiary --seeds 0 --out-root /content/smoke_out --results results/w5_smoke.jsonl --views-root /content/smoke_views 2>&1 | tail -10
import glob
from PIL import Image
from a1eval import load
rows = load('results/w5_smoke.jsonl')
need = {(m, k) for m in ('trellis2', 'hunyuan3d') for k in ('gen_ok', 'latency_s', 'dino_best', 'sil_iou_best', 'n_faces')}
got = {(r.model, r.metric) for r in rows if (r.metric != 'gen_ok' or r.value == 1.0)}
print('SMOKE PASSED' if need <= got else f'SMOKE FAILED, missing {sorted(need - got)}')
for f in sorted(glob.glob('/content/smoke_views/*/*.jpg')):
    print(f); display(Image.open(f))

## Cell D: full run (13 items x 3 seeds x 2 models = 78 meshes + 2 warm-ups)
Resumable: finished meshes are skipped, so after an interruption run the same cell again. Meshes go to `outputs/w5/` on Drive. Expect roughly 1-2 hours on the A100; watch the first few lines for per-mesh times and stop if one model is far slower than planned.

In [ ]:
!cd /content/TRELLIS.2 && /content/bin/micromamba run -n trellis2 env MAMBA_ROOT_PREFIX=/content/mamba python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_trellis2.py" 2>&1 | tee -a {BASE}/logs/w5_gen_trellis2.log | grep -v Warning | tail -50

In [ ]:
!cd /content/Hunyuan3D-2.1 && /content/bin/micromamba run -n hunyuan env MAMBA_ROOT_PREFIX=/content/mamba python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_hunyuan3d.py" 2>&1 | tee -a {BASE}/logs/w5_gen_hunyuan.log | grep -v Warning | tail -50

In [ ]:
!python pairs/w5_image_to_3d/score_w5.py --models trellis2 hunyuan3d --seeds 0 1 2 2>&1 | tee {BASE}/logs/w5_score.log | tail -20

## Cell E: tables, charts, qualitative grid
The grid shows the photographed objects, so keep *Omit code cell output when saving* ticked.

In [ ]:
!python pairs/w5_image_to_3d/report_w5.py --show 2>&1 | tee {BASE}/logs/w5_report.log | grep -v Warning
!mkdir -p {BASE}/report_assets && cp -r report_assets/w5 {BASE}/report_assets/
!/content/bin/micromamba run -n trellis2 pip freeze > {BASE}/results/w5_ENV_trellis2.txt
!/content/bin/micromamba run -n hunyuan pip freeze > {BASE}/results/w5_ENV_hunyuan.txt
!git -C /content/TRELLIS.2 rev-parse HEAD > {BASE}/results/w5_commit_trellis2.txt
!git -C /content/Hunyuan3D-2.1 rev-parse HEAD > {BASE}/results/w5_commit_hunyuan.txt
!cp /content/Hunyuan3D-2.1/LICENSE* {BASE}/results/ 2>/dev/null
from PIL import Image
display(Image.open('outputs/w5_figures/w5_qualitative.jpg'))

## Last cell: free the GPU
Set `AUTO_DISCONNECT = True` once everything has been copied to Drive.

In [ ]:
AUTO_DISCONNECT = False
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()